In [1]:
source("./scale.R")
test <- read.csv("/data/coro_data.csv")
test = test %>% rename(x = x_section, y = y_section)
library("xtable")
library("tibble")
options(warn=-1)

Loading required package: spatstat.data

Loading required package: spatstat.univar

spatstat.univar 3.1-1

Loading required package: spatstat.geom

spatstat.geom 3.3-4

Loading required package: spatstat.random

spatstat.random 3.3-2

Loading required package: spatstat.explore

Loading required package: nlme

spatstat.explore 3.3-3

Loading required package: spatstat.model

Loading required package: rpart

spatstat.model 3.3-3

Loading required package: spatstat.linnet

spatstat.linnet 3.2-3


spatstat 3.3-0 
For an introduction to spatstat, type ‘beginner’ 



Attaching package: ‘dplyr’


The following object is masked from ‘package:nlme’:

    collapse


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union




In [2]:
library('foreach')
library('doParallel')
cores=detectCores()
cl <- makeCluster(cores[1]-1) #not to overload your computer
registerDoParallel(cl)

Loading required package: iterators

Loading required package: parallel



### Feature calculation

In [3]:
ecdf_fun <- function(x,perc) ecdf(x)(perc)
## Calculate column stats for each type
calc_pp_2d<-function(typ,pp,section, rm){
    nnp <- c()
    nni <- c()
    mnn <- c()
    secs <- c()
    res<- list()
    ar <- 0
    cn <- 0
    for (i in 1:length(pp)){     
        perc<- ecdf_fun(nndist(pp[[i]]), 0.01)
        int<- intensity(pp[[i]])
        mval<- mean(marks(pp[[i]]))
        nnp<-append(nnp,perc)
        nni<-append(nni,int)
        mnn <- append(mnn, mval)
        secs<-append(secs, as.character(section[i]))
        ar <- ar + area(pp[[i]])
        cn <- cn + npoints(pp[[i]])
    }   
    res[['section']] = paste(secs, collapse = ', ')
    res[['area']] = round(ar,2)
    res[['perc']] = round(mean(nnp)*100,2)
    res[['lam']] = as.integer(round(mean(nni)))
    res[['conf']] = round(mean(mnn),2)
    res[['range']] = as.integer(round(1000*rm))
    res[['Cluster']] = typ
    res[['n']] = cn
    return(res)
}

### Excitatory clusters
We get the list of types and sections from the filtered table.

In [6]:
df<- read.csv("filter-inhib.csv")
df1 <- replace(df, is.na(df), 0)
df1[df1<=0.5] <- NA
df1<-df1[rowSums(is.na(df1)) != ncol(df1)-1, ]
row.names(df1) <- NULL

In [7]:
types = df1$cluster
types

[1] "0740 Pvalb Gaba_3" "0742 Pvalb Gaba_3" "0776 Sst Gaba_4"  
[4] "0798 Sst Gaba_9"   "0819 Sst Gaba_16"

In [8]:
sections<- apply(df1[,-1], 1, function(i) colnames(df1[,-1])[ !is.na(i) ]) ## qualified colns
sections<- sapply(sections, function(x){as.numeric(gsub("X.", "", x))})

In [16]:
df = data.frame(matrix(vector(), 0, 8,
                dimnames=list(c(), c('Cluster','section',"area",'range','perc', "lam",'conf','n'))),
                stringsAsFactors=F)
df$Cluster <- as.character(df$Cluster)
df$section <- as.character(df$section)
df

Cluster,section,area,range,perc,lam,conf,n
<chr>,<chr>,<lgl>,<lgl>,<lgl>,<lgl>,<lgl>,<lgl>


### Post region-selection  
We use the final window for each cluster and calculate the interaction range.

In [17]:
res <- foreach(t = 0:length(types), .combine=function(x,y) bind_rows(as.data.frame(x),as.data.frame(y)),.packages=c('dplyr','spatstat','poolr')) %dopar% {
    if (t == 0){
        q <- df
    }
    else{
        pp<-create_pp(test, NULL,sections[[t]], types[t])
        rm<-min(as.numeric(quantile(do.call(c,lapply(pp, nndist)),0.5)),0.12) 
        ## write into data frame
        q <- calc_pp_2d(types[t],pp,sections[[t]], rm)
        q
    }
}
res

Cluster,section,area,range,perc,lam,conf,n
<chr>,<chr>,<dbl>,<int>,<dbl>,<int>,<dbl>,<dbl>
0740 Pvalb Gaba_3,61,0.92,65,0,123,0.71,113
0742 Pvalb Gaba_3,61,1.29,81,0,63,0.71,82
0776 Sst Gaba_4,60,0.73,120,0,18,0.71,13
0798 Sst Gaba_9,59,1.32,120,0,13,0.70,17
0819 Sst Gaba_16,59,1.60,120,0,9,0.69,15


### Combine with aggregated p-values

In [11]:
pv <- read.csv("./table-in.csv")

In [12]:
pv<-add_column(pv, pv.adj= signif(p.adjust(pv$pval, method="BH"),2))
pv1<- pv[order(pv$pv.adj), ]
row.names(pv1) <- NULL
pv1

Cluster,X59,X60,X61,pval,perc,lam,conf,rm,pv.adj
<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<int>,<int>,<dbl>,<int>,<dbl>
0740 Pvalb Gaba_3,NA,NA,0.005,0.005,0,123,0.71,55,0.012
0742 Pvalb Gaba_3,NA,NA,0.005,0.005,0,63,0.71,72,0.012
0776 Sst Gaba_4,NA,0.01,NA,0.010,0,18,0.71,111,0.017
0819 Sst Gaba_16,0.019,NA,NA,0.019,0,9,0.69,111,0.024
0798 Sst Gaba_9,0.041,NA,NA,0.041,0,13,0.70,111,0.041


In [13]:
resp<-merge(pv[,c(1,5)],res,by="Cluster")
df<- resp[order(resp$pval), ]
row.names(df) <- NULL
df<-add_column(df, pv.adj= signif(p.adjust(df$pval, method="BH"),2), .after = 3)

In [14]:
drops <- c("pval")
df<-df[ , !(names(df) %in% drops)]
df

Cluster,section,pv.adj,area,range,perc,lam,conf,n
<chr>,<chr>,<dbl>,<dbl>,<int>,<dbl>,<int>,<dbl>,<dbl>
0740 Pvalb Gaba_3,61,0.012,0.92,60,0,123,0.71,113
0742 Pvalb Gaba_3,61,0.012,1.29,60,0,63,0.71,82
0776 Sst Gaba_4,60,0.017,0.73,60,0,18,0.71,13
0819 Sst Gaba_16,59,0.024,1.60,60,0,9,0.69,15
0798 Sst Gaba_9,59,0.041,1.32,60,0,13,0.70,17


In [32]:
print.xtable(xtable(df), file = "./feature-ex.txt")